# Detecting Outliers Using Visual Inspection and Simple Rules

**Milestone: Outlier Detection**  
**Project: Trivin Insight Engine**  
**Date: March 2026**

---

## Learning Objectives

By completing this notebook, you will be able to:

1. Identify outliers using boxplots and scatter plots
2. Apply simple rule-based checks (IQR and threshold)
3. Explain why values are flagged as potential outliers
4. Distinguish valid extremes from possible data errors
5. Avoid automatic removal and use context-aware reasoning

This milestone is detection-focused. No removal or modeling is required.

## Setup: Import Libraries and Load Data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print('Libraries imported successfully')

In [ ]:
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

csv_path = project_root / 'data' / 'raw' / 'employee_survey_2026_Q1.csv'
figure_dir = project_root / 'outputs' / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(csv_path, parse_dates=['survey_date'])
print(f'Dataset loaded from: {csv_path}')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')

numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
print('\nNumeric columns:')
print(numeric_columns)

df.head()

## Part 1: Understanding Outliers

Outliers are points that stand far from the majority of observations.
They can be:

- valid rare observations
- data collection or entry issues
- signs of a different subgroup

Outliers are context-dependent. Flagging is the first step, not final judgment.

In [ ]:
column = 'satisfaction_score'

fig, ax = plt.subplots()
ax.boxplot(df[column], vert=True, patch_artist=True,
           boxprops=dict(facecolor='lightsteelblue', edgecolor='navy'),
           medianprops=dict(color='darkred', linewidth=2),
           whiskerprops=dict(color='navy'),
           capprops=dict(color='navy'),
           flierprops=dict(marker='o', markerfacecolor='crimson', markersize=8, alpha=0.8))
ax.set_title('Boxplot: Satisfaction Score', fontsize=14, fontweight='bold')
ax.set_ylabel('Satisfaction Score', fontsize=12)
ax.set_xticklabels([column])
plt.tight_layout()
plt.savefig(figure_dir / 'notebook_outlier_boxplot_satisfaction_score.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved: outputs/figures/notebook_outlier_boxplot_satisfaction_score.png')

## Part 2: Visual Outlier Detection with Scatter Plot

Scatter plots help detect isolated points in two-variable space.
Use this to spot unusual combinations, not just unusual single values.

In [ ]:
x_col = 'work_life_balance'
y_col = 'satisfaction_score'

fig, ax = plt.subplots()
ax.scatter(df[x_col], df[y_col], color='steelblue', alpha=0.8, s=70, edgecolor='black', linewidth=0.4)
ax.set_xlabel('Work-Life Balance', fontsize=12)
ax.set_ylabel('Satisfaction Score', fontsize=12)
ax.set_title('Scatter Plot for Outlier Inspection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(figure_dir / 'notebook_outlier_scatter_work_life_vs_satisfaction.png', dpi=300, bbox_inches='tight')
plt.show()

print('Saved: outputs/figures/notebook_outlier_scatter_work_life_vs_satisfaction.png')

## Part 3: Detecting Outliers with Simple Rules

Rules support visual inspection.
They provide consistent flagging but do not decide automatic removal.

In [ ]:
col = 'satisfaction_score'
q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

iqr_flagged = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

print('IQR bounds:')
print(f'Lower bound: {lower_bound:.2f}')
print(f'Upper bound: {upper_bound:.2f}')
print(f'Flagged rows: {len(iqr_flagged)}')

iqr_flagged[['employee_id', 'department', col]].head()

In [ ]:
# Simple threshold for 1-10 scoring context
threshold_flagged = df[(df['satisfaction_score'] < 3) | (df['satisfaction_score'] > 9)]

print('Threshold rule: satisfaction_score < 3 or > 9')
print(f'Flagged rows: {len(threshold_flagged)}')

threshold_flagged[['employee_id', 'department', 'satisfaction_score']].head()

In [ ]:
iqr_ids = set(iqr_flagged['employee_id'].tolist())
threshold_ids = set(threshold_flagged['employee_id'].tolist())

print('Method overlap summary')
print(f'IQR only: {len(iqr_ids - threshold_ids)}')
print(f'Threshold only: {len(threshold_ids - iqr_ids)}')
print(f'Both methods: {len(iqr_ids.intersection(threshold_ids))}')

## Part 4: Interpretation and Reflection

Use these questions before deciding any action:

1. Are flagged points plausible real observations?
2. Could any flagged point be a data issue?
3. Which points are flagged by both visual and rule-based checks?
4. What additional context would you need before treatment?

### Video Walkthrough Checklist (~2 minutes)

- show one visual inspection step
- identify potential outliers
- apply one simple rule (IQR or threshold)
- explain why flagged points stand out
- state that no automatic removal is done

Reminder: detection comes before treatment.